# Demo: Research and Write an Article with LangChain

For comparison purposes, this notebook demonstrates how the same planner -> writer -> editor workflow from the CrewAI version can be implemented with LangChain chains.

## Preparing environment

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from time import perf_counter
from IPython.display import Markdown, display
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

In [3]:
AGENT_VERBOSE = True
TOPIC = "Artificial Intelligence"

## Prepare locally hosted LLM

In [4]:
llm = ChatOllama(
    model="mistral:latest",
    base_url="http://localhost:11434",
    temperature=0.7,
)
parser = StrOutputParser()

## Define Agent Roles

These prompts mirror the Planner, Writer, and Editor roles from the original notebook.

In [5]:
planner_system = (
    "You are a Content Planner. You are working on planning a blog article about {topic}. "
    "Collect information that helps the audience learn something and make informed decisions. "
    "Your output is the basis for the Content Writer."
 )

writer_system = (
    "You are a Content Writer. You are writing an opinion piece about {topic}. "
    "Base your writing on the planner output. Keep it factual, balanced, and engaging. "
    "Differentiate opinions from objective statements."
)

editor_system = (
    "You are an Editor. Review the writer draft to align with journalistic best practices, "
    "balanced viewpoints, and a consistent professional tone. Return polished markdown only."
)

## Define Tasks

In [6]:
plan_task = (
    "1. Prioritize the latest trends, key players, and noteworthy news on {topic}.\n"
    "2. Identify the target audience, considering their interests and pain points.\n"
    "3. Develop a detailed content outline including an introduction, key points, and a call to action.\n"
    "4. Include SEO keywords and relevant data or sources."
)

write_task = (
    "1. Use the content plan to craft a compelling blog post on {topic}.\n"
    "2. Incorporate SEO keywords naturally.\n"
    "3. Use engaging section titles.\n"
    "4. Structure the post with introduction, body, and conclusion.\n"
    "5. Keep language clear and publication-ready."
)

edit_task = (
    "Proofread the blog post for grammar, clarity, and tone alignment. "
    "Return final markdown ready for publication."
)

## Run Planner -> Writer -> Editor Pipeline

In [7]:
planner_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "{planner_system}"),
        ("human", "Task:\n{task}")
    ])
    | llm
    | parser
)

writer_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "{writer_system}"),
        ("human", "Task:\n{task}\n\nPlanner Output:\n{plan_output}")
    ])
    | llm
    | parser
)

editor_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "{editor_system}"),
        ("human", "Task:\n{task}\n\nDraft to edit:\n{draft_output}")
    ])
    | llm
    | parser
)

start = perf_counter()
plan_output = planner_chain.invoke({
    "planner_system": "".join(planner_system).format(topic=TOPIC),
    "task": plan_task.format(topic=TOPIC),
})

draft_output = writer_chain.invoke({
    "writer_system": "".join(writer_system).format(topic=TOPIC),
    "task": write_task.format(topic=TOPIC),
    "plan_output": plan_output,
})

final_output = editor_chain.invoke({
    "editor_system": "".join(editor_system),
    "task": "".join(edit_task),
    "draft_output": draft_output,
})
end = perf_counter()

if AGENT_VERBOSE:
    print(f"Planner output length: {len(plan_output)}")
    print(f"Writer output length: {len(draft_output)}")
    print(f"Editor output length: {len(final_output)}")

print(f"\n{'='*100}\nPipeline finished in {end - start:.2f} seconds.\n{'='*100}\n")

Planner output length: 2868
Writer output length: 2660
Editor output length: 2613

Pipeline finished in 24.88 seconds.



## Display Final Result as Markdown

In [8]:
markdown_text = final_output.strip()
if markdown_text.startswith("```"):
    lines = markdown_text.splitlines()
    if len(lines) > 2 and lines[-1].strip() == "```":
        markdown_text = "\n".join(lines[1:-1])
display(Markdown(markdown_text))

Title: Navigating the Future of Artificial Intelligence: Trends, Key Players, and Decision-Making Strategies

Welcome to our comprehensive guide on the dynamic world of Artificial Intelligence (AI). As technology continues to advance at an unprecedented pace, AI has emerged as a game-changer, shaping the future of innovation. In this article, we'll explore the latest trends defining the AI landscape, identify key players pushing these advancements forward, and provide valuable insights to help you make informed decisions about integrating AI into your business or personal endeavors.

**Latest Trends in Artificial Intelligence**

1. Quantum Computing for AI: Learn how quantum computing could revolutionize AI by processing vast amounts of data at unprecedented speeds. Examine the work of industry giants such as IBM, Google, and Microsoft in this exciting field.

2. Ethical AI and Data Privacy: Understand why ethical considerations are crucial in AI development and explore regulatory guidelines like GDPR and CCPA.

3. Edge AI and On-Device Learning: Discover the benefits of edge AI and on-device learning for real-time processing, and see how companies like Apple and Amazon are leveraging this technology.

**Key Players in the Artificial Intelligence Industry**

1. Google's DeepMind: Get an overview of DeepMind and its impressive achievements, such as AlphaGo and AlphaStar.

2. OpenAI: Discover OpenAI's mission for responsible AI development and learn about their innovative projects.

3. NVIDIA: Understand NVIDIA's role in powering the world's AI infrastructure.

**Noteworthy News in Artificial Intelligence**

1. Stay informed on recent breakthroughs in AI research, such as Google Brain's text-to-speech model.

2. Keep track of investments and acquisitions by major tech companies, like Microsoft's acquisition of Nuance Communications.

3. Follow government initiatives and partnerships aimed at advancing AI technologies.

**Making Informed Decisions about AI Integration**

1. Evaluating the Suitability of AI for Your Business or Project: Learn how to assess the feasibility of AI for your specific use case, taking factors like budget, resources, and project requirements into account.

2. Choosing the Right AI Solution Provider: Discover tips for selecting a reliable and effective partner, such as researching company history and assessing customer reviews.

Stay tuned as we delve deeper into the fascinating world of Artificial Intelligence. Subscribe to our newsletter for updates on the latest trends, insights, and strategies for harnessing AI's potential in your ventures.